# MCS603 — Data Analysis Module
## Notebook 2: Pandas — Data Wrangling, Imputation & Feature Engineering

---
### Learning Objectives
- Load, inspect, and clean a real-world dataset
- Handle missing values using multiple imputation strategies
- Reshape data: melt, pivot, pivot_table, stack/unstack
- Apply groupby aggregations and window functions
- Engineer new features: binning, encoding, ratios, date decomposition

### Outline
1. Loading & Profiling  
2. Missing Value Analysis  
3. Imputation Strategies  
4. Reshaping & Merging  
5. GroupBy & Aggregation  
6. Feature Engineering  
7. Exercises  

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.2f}".format)
np.random.seed(42)

In [ ]:
# Load dataset generated in Notebook 1
# If running standalone, the cell below recreates it
try:
    df = pd.read_csv("africa_health.csv")
    print("Loaded from africa_health.csv")
except FileNotFoundError:
    # Minimal recreation
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16 + ["East Africa"]*14 +
               ["North Africa"]*6 + ["Central Africa"]*8 + ["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           : countries,
        "region"            : regions,
        "year"              : 2022,
        "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
        "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
        "maternal_mortality": np.clip(np.random.exponential(350, N), 20, 2000),
        "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
        "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
        "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
        "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
        "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
        "malaria_incidence" : np.clip(np.random.exponential(120, N), 0, 450),
        "physicians_per1k"  : np.clip(np.random.exponential(0.8, N), 0.02, 5),
    })
    for col in ["maternal_mortality", "physicians_per1k", "malaria_incidence"]:
        df.loc[np.random.choice([True, False], N, p=[0.12, 0.88]), col] = np.nan

print(df.shape)
df.head(3)

---
## 1. Loading & Profiling

Always start with a systematic profiling pass before touching the data.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nDescriptive statistics:")
df.describe().round(2)

In [ ]:
def quick_profile(df):
    """Compact column-level summary."""
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "column"  : col,
            "dtype"   : str(s.dtype),
            "n_null"  : s.isna().sum(),
            "pct_null": f"{100*s.isna().mean():.1f}%",
            "n_unique": s.nunique(),
            "mean"    : s.mean() if pd.api.types.is_numeric_dtype(s) else "-",
            "min"     : s.min()  if pd.api.types.is_numeric_dtype(s) else s.min(),
            "max"     : s.max()  if pd.api.types.is_numeric_dtype(s) else s.max(),
        })
    return pd.DataFrame(rows).set_index("column")

quick_profile(df)

---
## 2. Missing Value Analysis

### Types of Missingness
| Type | Meaning | Imputation risk |
|---|---|---|
| **MCAR** | Missing Completely At Random | Low — safe to impute |
| **MAR** | Missing At Random (depends on observed data) | Moderate — conditional imputation needed |
| **MNAR** | Missing Not At Random (depends on missing value itself) | High — imputation may bias results |

In [ ]:
import matplotlib.pyplot as plt

missing = df.isnull()
miss_pct = (missing.sum() / len(df) * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
miss_pct.plot.bar(ax=axes[0], color="#E74C3C", alpha=0.8)
axes[0].set_title("Missing Values by Column (%)", fontsize=11)
axes[0].set_ylabel("%")
axes[0].tick_params(axis="x", rotation=30)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.1f}%",
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha="center", fontsize=9)

# Heatmap of missingness pattern
cols_with_na = df.columns[df.isnull().any()].tolist()
axes[1].imshow(df[cols_with_na].isnull().T.astype(int),
               aspect="auto", cmap="Reds", interpolation="nearest")
axes[1].set_yticks(range(len(cols_with_na)))
axes[1].set_yticklabels(cols_with_na, fontsize=9)
axes[1].set_xlabel("Row index")
axes[1].set_title("Missingness Pattern (red = missing)", fontsize=11)

plt.tight_layout()
plt.savefig("missing_values.png", bbox_inches="tight")
plt.show()

---
## 3. Imputation Strategies

| Strategy | Best for | Risk |
|---|---|---|
| Mean | Symmetric, no outliers | Reduces variance |
| Median | Skewed data / outliers present | Robust |
| Mode | Categorical | May over-represent one value |
| Forward/Backward fill | Time-series | Assumes temporal continuity |
| Group-conditional | MAR data | Best balance |
| Regression/KNN | Complex relationships | Computationally expensive |

In [ ]:
df_work = df.copy()
col = "maternal_mortality"

# Strategy 1: Global mean
df_mean = df_work[col].fillna(df_work[col].mean())

# Strategy 2: Global median
df_median = df_work[col].fillna(df_work[col].median())

# Strategy 3: Region-conditional median (best for MAR)
df_work["maternal_mortality_imputed"] = df_work.groupby("region")[col]\
    .transform(lambda x: x.fillna(x.median()))

print("Imputation Comparison for maternal_mortality:")
print(f"  Original    mean  = {df[col].mean():.1f}  std = {df[col].std():.1f}")
print(f"  Mean imp.   mean  = {df_mean.mean():.1f}  std = {df_mean.std():.1f}")
print(f"  Median imp. mean  = {df_median.mean():.1f}  std = {df_median.std():.1f}")
print(f"  Region cond mean  = {df_work['maternal_mortality_imputed'].mean():.1f}  "
          f"std = {df_work['maternal_mortality_imputed'].std():.1f}")

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, (label, data, color) in zip(axes, [
    ("Mean Imputation",   df_mean,   "#3498DB"),
    ("Median Imputation", df_median, "#2ECC71"),
    ("Region-Conditional Median", df_work["maternal_mortality_imputed"], "#E67E22"),
]):
    ax.hist(data, bins=20, color=color, alpha=0.7, edgecolor="white", label=label)
    ax.axvline(data.mean(), color="black", linestyle="--")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Maternal Mortality")

plt.suptitle("Imputation Strategy Comparison", fontsize=12)
plt.tight_layout()
plt.savefig("imputation_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Apply best strategy to all columns and produce a clean dataset
df_clean = df.copy()

imputation_plan = {
    "maternal_mortality": ("region_median", "region"),
    "physicians_per1k"  : ("region_median", "region"),
    "malaria_incidence" : ("region_median", "region"),
}

for col, (strategy, group_col) in imputation_plan.items():
    before = df_clean[col].isna().sum()
    if strategy == "region_median":
        df_clean[col] = df_clean.groupby(group_col)[col]\
            .transform(lambda x: x.fillna(x.median()))
    after = df_clean[col].isna().sum()
    print(f"{col:<25}: {before} NaN → {after} NaN  [{strategy}]")

print(f"\nRemaining NaNs: {df_clean.isnull().sum().sum()}")

---
## 4. Reshaping & Merging

### 4.1 melt — Wide → Long

In [ ]:
# melt: convert multiple metric columns into rows
metrics = ["infant_mortality", "maternal_mortality", "hiv_prevalence", "malaria_incidence"]
df_long  = df_clean.melt(
    id_vars   = ["country", "region"],
    value_vars= metrics,
    var_name  = "indicator",
    value_name= "value"
)
print("Wide shape:", df_clean.shape)
print("Long shape: ", df_long.shape)
df_long.head(8)

### 4.2 pivot_table — Long → Wide with aggregation

In [ ]:
# Mean of each indicator by region
pivot = df_long.pivot_table(
    index   = "region",
    columns = "indicator",
    values  = "value",
    aggfunc = "mean"
).round(1)
pivot

### 4.3 Merging DataFrames

In [ ]:
# Simulate a second table: regional population data
pop_data = pd.DataFrame({
    "region"     : ["West Africa","East Africa","North Africa",
                    "Central Africa","Southern Africa"],
    "population_m": [420, 455, 245, 175, 68],
    "area_km2_k" : [5112, 6363, 8525, 5040, 2693],
})

# Left join: keep all rows from df_clean
df_merged = df_clean.merge(pop_data, on="region", how="left")
print("After merge:", df_merged.shape)
df_merged[["country","region","population_m","area_km2_k"]].head(6)

---
## 5. GroupBy & Aggregation

In [ ]:
# Multiple aggregations in one call
region_stats = df_clean.groupby("region").agg(
    countries        = ("country",          "count"),
    mean_life_exp    = ("life_expectancy",   "mean"),
    mean_infant_mort = ("infant_mortality",  "mean"),
    mean_gdp         = ("gdp_per_capita",    "mean"),
    mean_vaccination = ("vaccination_dtp3",  "mean"),
    mean_hiv         = ("hiv_prevalence",    "mean"),
).round(1)

region_stats

In [ ]:
# transform: add group-level stats back to original rows
df_clean["region_mean_le"]   = df_clean.groupby("region")["life_expectancy"].transform("mean")
df_clean["le_vs_region_mean"]= df_clean["life_expectancy"] - df_clean["region_mean_le"]

# Countries most above their regional average
df_clean.nlargest(5, "le_vs_region_mean")[["country","region","life_expectancy","region_mean_le","le_vs_region_mean"]]

In [ ]:
# apply: custom function per group
def top_n(group, col, n=2):
    return group.nlargest(n, col)[["country", col]]

print("Top 2 countries by life expectancy in each region:")
result = df_clean.groupby("region", group_keys=False).apply(
    lambda g: top_n(g, "life_expectancy", 2)
)
print(result.to_string())

---
## 6. Feature Engineering

Feature engineering transforms raw variables into representations more useful for analysis and modelling.

In [ ]:
# 6.1 Binning — continuous → categorical
df_clean["life_exp_band"] = pd.cut(
    df_clean["life_expectancy"],
    bins   = [0, 55, 65, 75, 100],
    labels = ["Low", "Moderate", "High", "Very High"],
    right  = True
)

# Percentile-based bins
df_clean["gdp_quartile"] = pd.qcut(
    df_clean["gdp_per_capita"],
    q      = 4,
    labels = ["Q1","Q2","Q3","Q4"]
)

print("Life expectancy bands:")
print(df_clean["life_exp_band"].value_counts())
print("\nGDP quartiles:")
print(df_clean["gdp_quartile"].value_counts().sort_index())

In [ ]:
# 6.2 Encoding — categorical → numeric

# Label encoding
df_clean["region_code"] = df_clean["region"].astype("category").cat.codes

# One-hot encoding
region_dummies = pd.get_dummies(df_clean["region"], prefix="reg", drop_first=True)
print("One-hot encoded columns:", region_dummies.columns.tolist())

# Ordinal encoding for life_exp_band
ordinal_map = {"Low": 1, "Moderate": 2, "High": 3, "Very High": 4}
df_clean["life_exp_ordinal"] = df_clean["life_exp_band"].map(ordinal_map)

In [ ]:
# 6.3 Ratio & composite features
df_clean["health_gdp_ratio"]     = df_clean["health_expenditure"] / df_clean["gdp_per_capita"] * 1000
df_clean["child_survival_rate"]  = 1000 - df_clean["infant_mortality"]
df_clean["log_gdp"]              = np.log1p(df_clean["gdp_per_capita"])
df_clean["log_maternal_mort"]    = np.log1p(df_clean["maternal_mortality"])

# Health composite index (0-100 scale)
cols_to_norm = ["life_expectancy", "vaccination_dtp3", "water_access"]
for c in cols_to_norm:
    mn, mx = df_clean[c].min(), df_clean[c].max()
    df_clean[f"{c}_norm"] = (df_clean[c] - mn) / (mx - mn)

df_clean["health_index"] = (
    df_clean["life_expectancy_norm"] * 0.5 +
    df_clean["vaccination_dtp3_norm"] * 0.3 +
    df_clean["water_access_norm"] * 0.2
) * 100

df_clean[["country", "region", "health_index"]]\
    .sort_values("health_index", ascending=False).head(10)

In [ ]:
# 6.4 Standardisation & Normalisation
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if not c.endswith("_norm")]

# Z-score standardisation  (mean=0, std=1)
df_zscore = (df_clean[numeric_cols] - df_clean[numeric_cols].mean()) / df_clean[numeric_cols].std()

# Min-max normalisation  (0–1)
df_minmax = (df_clean[numeric_cols] - df_clean[numeric_cols].min()) / \
            (df_clean[numeric_cols].max() - df_clean[numeric_cols].min())

print("Z-score standardised (first 3 cols):")
print(df_zscore.iloc[:3, :5].round(3))

In [ ]:
# Correlation matrix of engineered features
feat_cols = ["life_expectancy", "infant_mortality", "gdp_per_capita",
             "health_expenditure", "vaccination_dtp3", "water_access",
             "hiv_prevalence", "health_index", "log_gdp"]

corr = df_clean[feat_cols].corr().round(2)
print("Correlation Matrix (selected features):")
corr

---
## 7. Exercises

### Exercise 1: Missing Value Strategy
For `physicians_per1k`:  
a) Create three imputed versions: global mean, global median, country income quartile median  
b) Compare the distributions using histograms  
c) Which strategy would you recommend and why?

In [ ]:
# Exercise 1


### Exercise 2: GroupBy Analysis
Group by `life_exp_band` and compute:  
- Average GDP per capita  
- Average health expenditure  
- Average vaccination rate  
- Number of countries  

Sort by life expectancy band (Low → Very High) and print as a formatted table.

In [ ]:
# Exercise 2


### Exercise 3: Feature Engineering
Create a `disease_burden` composite score as:  
```
disease_burden = 0.4 × norm(infant_mortality)
               + 0.35 × norm(maternal_mortality)
               + 0.25 × norm(hiv_prevalence)
```
Scale to 0–100. List the 5 most and 5 least burdened countries.

In [ ]:
# Exercise 3


In [ ]:
# Save enriched dataset for later notebooks
df_clean.to_csv("africa_health_clean.csv", index=False)
print("Saved africa_health_clean.csv")
print("Columns:", df_clean.columns.tolist())

---
## Summary

| Task | Pandas Tool |
|---|---|
| Profile columns | `describe()`, custom `quick_profile()` |
| Detect NaNs | `isna().sum()`, heatmap |
| Global imputation | `fillna(mean/median)` |
| Group imputation | `groupby().transform()` |
| Wide → Long | `melt()` |
| Aggregated pivot | `pivot_table()` |
| Join tables | `merge(how="left/inner/outer")` |
| Group stats | `groupby().agg()` |
| Binning | `pd.cut()` (equal width), `pd.qcut()` (quantiles) |
| Encoding | `get_dummies()`, `.cat.codes` |
| Standardise | `(x - mean) / std` |

**Next:** Notebook 3 — Matplotlib/Seaborn Visualisation